# Solar Filament Segmentation Challenge 2026 — full pipeline

This notebook covers the whole pipeline in two parts.

* **Part A — training (optional, `RUN_TRAINING = True`).** Data preparation and folds → 10 training runs (5 folds × 2 seeds, about 8 h each) → SWA of seed 1 → out‑of‑fold (OOF) predictions → choice of α, threshold, area and retain fraction by nested CV. It is off by default: the cells print the commands and skip, and Part B uses the released weights.
* **Part B — inference.** It regenerates the submitted predictions (`release/submission_final.csv`) from the 10 weights (5 folds × {seed‑1 SWA, seed 2}): the released ones by default, or the ones trained in Part A. It runs

`16‑view TTA per fold → vote score (α=0.75) → 5‑fold fusion → post‑processing (thr 0.32, area ≥ 200, retain fraction ≥ 0.8) → CSV`.

Set `N_IMAGES = None` in the settings cell for the full run (about 10 h on one 16 GB GPU, 41 s per test image). The default `N_IMAGES = 3` runs the same code on 3 test images (about 10 minutes) as a check. Requirements: `requirements.txt`, detectron2 0.6 and Mask DINO with `patches/maskdino_filament.patch` (see `docs/REPRODUCE.md`).

In [ ]:
import os, sys, json, hashlib, subprocess
from pathlib import Path
REPO = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(REPO); sys.path.insert(0, str(REPO / "src"))
RUN_TRAINING = False  # True = Part A retrains everything (~80 h GPU) and Part B uses those weights
N_IMAGES = 3          # None = all test images (full reproduction, ~10 h)
ALPHA, SCORE_THR, MIN_AREA, KEEP_FRAC = 0.75, 0.32, 200, 0.8
WD, OUT, WORK = REPO / "release" / "weights", REPO / "release" / "run", REPO / "work_dirs"
OUT.mkdir(parents=True, exist_ok=True)
if RUN_TRAINING: WORK.mkdir(exist_ok=True)
def run(cmd, train_only=False):
    rel = lambda c: os.path.relpath(c, REPO) if isinstance(c, Path) else str(c)
    print(">", " ".join(rel(c) for c in cmd))
    if train_only and not RUN_TRAINING:
        print("  (skipped: RUN_TRAINING = False)"); return
    subprocess.run([str(c) for c in cmd], check=True)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
TTA = ["--tta", "--tta-views", 8, "--tta-merge", "prob", "--mask-bin-thr", 0.5]   # 2 models × 8 D4 views = 16 views

# Part A — training (optional)
Off by default (`RUN_TRAINING = False`): each cell prints its commands and skips. The same steps are in `docs/REPRODUCE.md`. Hardware used: one RTX 5080 (16 GB).

## A1. Data and folds
`prepare_data.py` converts the annotations to a single class (`data/magfilo_singleclass.json`) and splits the photos into 5 folds with StratifiedGroupKFold (group = photo, strata = GONG station × solar‑cycle bin, seed 42). The cell checks that the regenerated `data/folds.csv` is identical to the one in the repository.

In [ ]:
if RUN_TRAINING:
    ref = (REPO / "data" / "folds.csv").read_bytes()
    run(["python3", "src/prepare_data.py", "--k", 5, "--seed", 42])
    assert (REPO / "data" / "folds.csv").read_bytes() == ref, "folds.csv differs from the released fold split"
    print("folds.csv identical to the released split")
else:
    run(["python3", "src/prepare_data.py", "--k", 5, "--seed", 42], train_only=True)

## A2. Training: 5 folds × 2 seeds (about 8 h per run)
Mask DINO R‑50 at native 2048 px, 5000 iterations. Seed 1 uses `configs/maskdino_filament_r50_full2048.yaml` (fold 0) and `configs/_s3_f{k}.yaml` (folds 1–4); seed 2 uses `configs/_seed2_f{k}.yaml`. The configs differ only in the fold. `train_with_watchdog.sh` resumes from the last checkpoint (saved every 250 iterations) if a run stops, and each run aborts if its training record count does not match the fold (928, 939, 920, 915, 914 for folds 0–4).

In [ ]:
for k in range(5):
    cfg1 = "configs/maskdino_filament_r50_full2048.yaml" if k == 0 else f"configs/_s3_f{k}.yaml"
    run(["bash", "scripts/train_with_watchdog.sh", cfg1, WORK / f"r50_full2048_f{k}",
         WORK / f"train_f{k}.log", 5000, k], train_only=True)
    run(["bash", "scripts/train_with_watchdog.sh", f"configs/_seed2_f{k}.yaml", WORK / f"r50_full2048_seed2_f{k}",
         WORK / f"train_seed2_f{k}.log", 5000, k], train_only=True)

## A3. SWA of seed 1
The last four checkpoints of each seed‑1 run (iterations 4249, 4499, 4749, 4999) are averaged into `model_swaA.pth`. The backbone uses FrozenBN and the heads GroupNorm, so no BN statistics need recomputing.

In [ ]:
for k in range(5):
    run(["python3", "src/swa_average.py", "--run-dir", WORK / f"r50_full2048_f{k}",
         "--iters", 4249, 4499, 4749, 4999, "--out", WORK / f"r50_full2048_f{k}" / "model_swaA.pth"], train_only=True)
TRAINED = {(k, "seed1_swa"): WORK / f"r50_full2048_f{k}" / "model_swaA.pth" for k in range(5)}
TRAINED.update({(k, "seed2"): WORK / f"r50_full2048_seed2_f{k}" / "model_final.pth" for k in range(5)})

## A4. Out‑of‑fold predictions
Each fold's two models predict their own held‑out photos with the same 16‑view TTA as the test run. The 5 folds together cover all 707 training photos (OOF‑707). The vote score with α = 0.75 is then applied; the result corresponds to `release/oof_swa2_a0.75.json`.

In [ ]:
oof = {}
for k in range(5):
    raw = WORK / f"r50_full2048_seed2_f{k}" / "val_raw_prob2m_swa_votes.json"
    run(["python3", "-u", "src/predict_submit.py", "--config", f"configs/_seed2_f{k}.yaml",
         "--weights", TRAINED[(k, "seed1_swa")], TRAINED[(k, "seed2")], *TTA,
         "--split", "val", "--fold", k, "--dump-raw", raw, "--out", WORK / f"oof_f{k}.csv"], train_only=True)
    if RUN_TRAINING:
        oof.update(json.loads(raw.read_text())["preds"])
if RUN_TRAINING:
    (WORK / "oof_swa2_raw.json").write_text(json.dumps({"preds": oof}))
    print(len(oof), "OOF photos")
run(["python3", "src/vote_rescore.py", "--preds", WORK / "oof_swa2_raw.json", "--alpha", ALPHA,
     "--out", WORK / "oof_swa2_a0.75.json"], train_only=True)

## A5. Choice of post‑processing (nested CV)
`pp_fine.py` searches α × score threshold × minimum area (275 combinations); `pp_retain.py` adds the retain fraction (threshold × area × fraction, 120 combinations). Both score the search with nested 5‑fold CV on OOF‑707. The released choice is α 0.75, threshold 0.32, area 200, retain fraction 0.8 (picked in all 5 outer folds; `release/postproc_final.json`).

In [ ]:
run(["python3", "-u", "scripts/pp_fine.py", "--out", WORK / "pp_fine.json"], train_only=True)
run(["python3", "-u", "scripts/pp_retain.py", 8, WORK / "pp_retain.npz"], train_only=True)
print(json.dumps({k: v for k, v in json.loads((REPO / "release" / "postproc_final.json").read_text()).items()
                  if k in ("score_thr", "min_area", "min_keep_frac", "val_pq")}))

# Part B — inference: test predictions to the submission CSV

## 1. Weights
With `RUN_TRAINING = False` the released weights are used. Each `.pth` is stored in parts below GitHub's 100 MB limit; `assemble` joins them and checks the SHA‑256 from `MANIFEST.json`. With `RUN_TRAINING = True` the weights from Part A are used instead.

In [ ]:
if RUN_TRAINING:
    W = TRAINED
else:
    from weights_io import assemble
    W = {(k, s): assemble(WD, f"fold{k}_{s}.pth") for k in range(5) for s in ("seed1_swa", "seed2")}
print(len(W), "weight files", "(trained in Part A)" if RUN_TRAINING else "verified")

## 2. Test inference, per fold
Two models per fold × 8 D4 views = 16 views, merged by probability averaging with a topology guard; the number of agreeing views is stored with each instance (`--dump-raw`).

In [ ]:
raws = []
for k in range(5):
    raw = OUT / f"test_raw_f{k}.json"
    if not raw.exists():
        cmd = ["python3", "-u", "src/predict_submit.py", "--config", f"configs/_seed2_f{k}.yaml",
               "--weights", W[(k, "seed1_swa")], W[(k, "seed2")],
               *TTA,
               "--dump-raw", raw, "--out", OUT / f"tmp_f{k}.csv"]
        if N_IMAGES: cmd += ["--limit", N_IMAGES]
        subprocess.run([str(c) for c in cmd])      # exit code 1 only means the partial CSV failed the full-set check
    raws.append(raw)

## 3. Vote score and five‑fold fusion
`score' = score · (votes / 16)^α`; fold outputs are clustered (IoU > 0.5), a cluster needs ≥ 2 of 5 folds, pixels are merged by majority.

In [ ]:
per = []
for k, raw in enumerate(raws):
    o = OUT / f"test_raw_f{k}_a{ALPHA}.json"
    run(["python3", "src/vote_rescore.py", "--preds", raw, "--alpha", ALPHA, "--out", o]); per.append(o)
run(["python3", "-u", "src/ensemble.py", "--preds", *per, "--out", OUT / "test_fused.json", "--min-votes", 2])

## 4. Post‑processing and CSV
Instances are painted in descending score order (pixel‑exclusive); kept if score ≥ 0.32, painted area ≥ 200 px and at least 80 % of its own pixels are still unclaimed (retain fraction; drops fragments of already‑kept filaments). `json_to_csv.py` also validates the CSV.

In [ ]:
(OUT / "postproc.json").write_text(json.dumps({"score_thr": SCORE_THR, "min_area": MIN_AREA, "mask_thr": 0.5, "min_keep_frac": KEEP_FRAC}))
run(["python3", "src/json_to_csv.py", "--preds", OUT / "test_fused.json", "--postproc", OUT / "postproc.json", "--out", OUT / "submission.csv"])
md5 = lambda p: hashlib.md5(Path(p).read_bytes()).hexdigest()
print("reproduced:", md5(OUT / "submission.csv"))
print("submitted :", md5(REPO / "release" / "submission_final.csv"))
if N_IMAGES: print("(partial run: the hashes differ by design; set N_IMAGES = None for the full comparison)")

## 5. Out‑of‑fold score of this configuration
`release/oof_swa2_a0.75.json` (or Part A's `work_dirs/oof_swa2_a0.75.json`) holds the OOF predictions (the same pipeline applied to each fold's held‑out photos) that were used to choose α, the threshold and the area. The cell computes the pooled PQ with the organisers' definition (`src/pq_metric.py`); it needs the competition's training data (`FILAMENT_ROOT`) and writes `data/magfilo_singleclass.json` first if missing.

In [ ]:
if not (REPO / "data" / "magfilo_singleclass.json").exists():
    run(["python3", "src/prepare_data.py"])
OOF_JSON = WORK / "oof_swa2_a0.75.json" if RUN_TRAINING else REPO / "release" / "oof_swa2_a0.75.json"
run(["python3", "-u", "src/tune_postproc.py", "--preds", OOF_JSON, "--fold", -1,
     "--score-thr", SCORE_THR, "--min-area", MIN_AREA, "--min-keep-frac", KEEP_FRAC, "--out", OUT / "oof_eval.json"])
print({k: round(v, 4) for k, v in json.loads((OUT / "oof_eval.json").read_text()).items() if k in ("val_pq", "val_sq", "val_rq")})
# expected: val_pq 0.4777, val_sq 0.689, val_rq 0.693